# Variational Low-Rank Adaptation Using IVON
## Qwen2.5-0.5B reproduction on ARC-Easy
**Team:** Siddhu, Pathlavath Shiva Kumar, Gugulothu Ganesh.

Select **Runtime > Change runtime type > T4 GPU**, then **Run all**.
This notebook contains a readable snapshot of the project source. It works without GitHub authentication.
The first large cell only writes the displayed source files into `/content/ivon_lora_study`.

**Evidence rule:** the notebook computes results from real training. An unexecuted notebook contains no measured results.
The two-step smoke test checks the pipeline; it cannot establish a calibration benefit.

Sources: [paper](https://arxiv.org/abs/2411.04421), [official code](https://github.com/team-approx-bayes/ivon-lora), [Qwen model](https://huggingface.co/Qwen/Qwen2.5-0.5B), [ARC data](https://huggingface.co/datasets/allenai/ai2_arc).


In [ ]:
from pathlib import Path
import os, json, subprocess, sys
ROOT = Path('/content/ivon_lora_study')
ROOT.mkdir(parents=True, exist_ok=True)
FILES = {
'configs/quick.json': r'''{
  "model_id": "Qwen/Qwen2.5-0.5B",
  "dataset_id": "allenai/ai2_arc",
  "dataset_config": "ARC-Easy",
  "seeds": [21, 42, 87],
  "data_seed": 2026,
  "train_size": 512,
  "validation_size": 128,
  "test_size": 256,
  "max_length": 256,
  "steps": 128,
  "batch_size": 4,
  "accumulation_steps": 2,
  "eval_batch_size": 8,
  "lora_rank": 8,
  "lora_alpha": 16,
  "lora_dropout": 0.0,
  "target_modules": ["q_proj", "v_proj"],
  "adamw_lr": 0.0002,
  "ivon_lr": 0.03,
  "weight_decay": 0.0001,
  "ess": 1000000,
  "hess_init": 0.001,
  "clip_radius": 0.001,
  "beta2": 0.99999,
  "warmup_fraction": 0.1,
  "posterior_samples": 10,
  "ece_bins": 15,
  "dtype": "float32",
  "save_checkpoints": true
}
''',
'configs/registered_t4.json': r'''{
  "model_id": "Qwen/Qwen2.5-0.5B",
  "dataset_id": "allenai/ai2_arc",
  "dataset_config": "ARC-Easy",
  "seeds": [
    21,
    42,
    87
  ],
  "data_seed": 2026,
  "train_size": 512,
  "validation_size": 128,
  "test_size": 256,
  "max_length": 256,
  "steps": 128,
  "batch_size": 4,
  "accumulation_steps": 2,
  "eval_batch_size": 8,
  "lora_rank": 8,
  "lora_alpha": 16,
  "lora_dropout": 0.0,
  "target_modules": [
    "q_proj",
    "v_proj"
  ],
  "adamw_lr": 0.0002,
  "ivon_lr": 0.03,
  "weight_decay": 0.0001,
  "ess": 1000000,
  "hess_init": 0.001,
  "clip_radius": 0.001,
  "beta2": 0.99999,
  "warmup_fraction": 0.1,
  "posterior_samples": 10,
  "ece_bins": 15,
  "dtype": "float32",
  "save_checkpoints": true,
  "model_revision": "060db6499f32faf8b98477b0a26969ef7d8b9987",
  "dataset_revision": "210d026faf9955653af8916fad021475a3f00453",
  "source_sha256": "8318bc20eae8eda7a87506110ebc099296399b2f36e86878b2c11c7b6a097188"
}''',
'configs/smoke.json': r'''{
  "extends": "quick.json",
  "seeds": [21],
  "train_size": 16,
  "validation_size": 8,
  "test_size": 8,
  "steps": 2,
  "posterior_samples": 2,
  "save_checkpoints": false
}
''',
'configs/standard.json': r'''{
  "extends": "quick.json",
  "train_size": 1024,
  "validation_size": 256,
  "test_size": 512,
  "steps": 256
}
''',
'pyproject.toml': r'''[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "ivon-lora-study"
version = "1.0.0"
description = "A small-scale, auditable Qwen reproduction of IVON-LoRA"
requires-python = ">=3.10"

[tool.setuptools.packages.find]
where = ["src"]

[tool.pytest.ini_options]
testpaths = ["tests"]
pythonpath = ["src"]
''',
'requirements.txt': r'''# Colab already provides CUDA-enabled PyTorch. Do not replace it with a CPU wheel.
torch>=2.5,<2.9
transformers==4.55.4
peft==0.17.1
datasets==4.0.0
accelerate==1.10.1
ivon-opt==0.1.3
numpy>=1.26,<3
matplotlib==3.10.6
pytest==8.4.2
''',
'scripts/training_figures.py': r'''"""Plot recorded training losses and tabulate measured runtime/adapter details."""
import argparse
import csv
import json
from pathlib import Path


def make_training_figures(root):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    root = Path(root)
    config = json.loads((root / 'config.json').read_text())
    seeds = config['seeds']
    fig, axes = plt.subplots(1, len(seeds), figsize=(4.4 * len(seeds), 3.8),
                             squeeze=False, constrained_layout=True)
    rows = []
    for ax, seed in zip(axes[0], seeds):
        for method, color in [('adamw', '#43566e'), ('ivon', '#108078')]:
            folder = root / f'seed_{seed}' / method
            info = json.loads((folder / 'complete.json').read_text())
            history = json.loads((folder / 'training_history.json').read_text())
            ax.plot([h['step'] for h in history], [h['loss'] for h in history],
                    label=method, color=color, linewidth=1.3, alpha=.85)
            rows.append({'seed': seed, 'optimizer': method, 'steps': info['steps'],
                         'train_seconds': info['train_seconds'],
                         'seconds_per_step': info['train_seconds'] / info['steps'],
                         'peak_train_memory_gib': info['peak_train_memory_gib'],
                         'trainable_parameters': info['trainable_parameters'],
                         'total_parameters': info['total_parameters'],
                         'first_batch_ce': history[0]['loss'],
                         'last_batch_ce': history[-1]['loss']})
        ax.set(title=f'Seed {seed}', xlabel='Optimizer step', ylabel='Training batch cross-entropy')
        ax.spines[['top', 'right']].set_visible(False)
        ax.legend()
    fig.suptitle('Recorded training losses\nIVON evaluates a sampled adapter; these curves exclude KL regularization.', fontsize=11)
    dest = root / 'summary'
    dest.mkdir(exist_ok=True)
    fig.savefig(dest / 'training_curves.png', dpi=180)
    plt.close(fig)
    with (dest / 'training_details.csv').open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    print('Saved:', dest / 'training_curves.png')
    print('Saved:', dest / 'training_details.csv')
    return rows


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('run_dir')
    make_training_figures(parser.parse_args().run_dir)
''',
'src/ivon_lora/__init__.py': r'''"""Auditable IVON-LoRA reproduction. No experiment runs on import."""
__version__ = "1.0.0"
''',
'src/ivon_lora/data.py': r'''"""Fixed ARC subsets, robust answer mapping, and explicit exclusions."""
import hashlib
import json
import random
from collections import Counter

LETTERS = "ABCD"


def format_arc(row):
    labels = [str(x) for x in row["choices"]["label"]]
    texts = row["choices"]["text"]
    if len(labels) != 4 or len(texts) != 4 or len(set(labels)) != 4:
        raise ValueError("Exactly four distinct option labels are required")
    key = str(row["answerKey"])
    if key not in labels:
        raise ValueError("Answer key is absent from the option labels")
    choices = "\n".join(f"{letter}. {text}" for letter, text in zip(LETTERS, texts))
    # Ground truth is deliberately absent from the input prompt.
    prompt = f"Select the correct answer.\nQuestion: {row['question']}\n{choices}\nAnswer:"
    return {"id": str(row["id"]), "prompt": prompt, "label": labels.index(key)}


def prepare_splits(raw, tokenizer, config):
    result, manifest = {}, {"data_seed": config["data_seed"], "splits": {}}
    for split in ["train", "validation", "test"]:
        usable, excluded = [], Counter()
        seen = set()
        for row in raw[split]:
            try:
                item = format_arc(row)
            except ValueError:
                excluded["invalid_or_non_four_choice"] += 1
                continue
            if item["id"] in seen:
                raise ValueError(f"Duplicate id in {split}: {item['id']}")
            seen.add(item["id"])
            ids = tokenizer.encode(item["prompt"], add_special_tokens=False)
            if len(ids) > config["max_length"]:
                excluded["over_max_length"] += 1
                continue
            item["input_ids"] = ids
            usable.append(item)
        random.Random(config["data_seed"]).shuffle(usable)
        count = config[f"{split}_size"]
        if count > len(usable) or count <= 0:
            raise ValueError(f"Requested {count} {split} items, only {len(usable)} eligible")
        selected = usable[:count]
        result[split] = selected
        raw_json = json.dumps(selected, sort_keys=True, ensure_ascii=False)
        manifest["splits"][split] = {
            "original_count": len(raw[split]), "eligible_count": len(usable),
            "excluded": dict(excluded), "selected_count": count,
            "ids": [r["id"] for r in selected],
            "label_counts": dict(Counter(r["label"] for r in selected)),
            "sha256": hashlib.sha256(raw_json.encode()).hexdigest()}
    for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
        if set(manifest["splits"][a]["ids"]) & set(manifest["splits"][b]["ids"]):
            raise ValueError(f"Overlapping ids in {a} and {b}")
        hashes_a = {hashlib.sha256(r["prompt"].encode()).hexdigest() for r in result[a]}
        hashes_b = {hashlib.sha256(r["prompt"].encode()).hexdigest() for r in result[b]}
        if hashes_a & hashes_b:
            raise ValueError(f"Duplicate prompts across {a} and {b}")
    return result, manifest


class Collator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def __call__(self, rows):
        import torch
        width = max(len(r["input_ids"]) for r in rows)
        # Right padding and explicit last-valid-token indexing avoid padding bugs.
        ids = [r["input_ids"] + [self.pad_token_id] * (width - len(r["input_ids"])) for r in rows]
        masks = [[1] * len(r["input_ids"]) + [0] * (width - len(r["input_ids"])) for r in rows]
        return {"input_ids": torch.tensor(ids), "attention_mask": torch.tensor(masks),
                "labels": torch.tensor([r["label"] for r in rows])}
''',
'src/ivon_lora/metrics.py': r'''"""Multiclass calibration metrics. Probabilities and ECE use the [0, 1] scale."""
import numpy as np


def validate(probs, labels):
    p = np.asarray(probs, dtype=np.float64)
    y = np.asarray(labels)
    if p.ndim != 2 or p.shape[0] == 0 or p.shape[1] < 2:
        raise ValueError("Expected nonempty N by C probabilities, with C >= 2")
    if y.shape != (len(p),) or not np.issubdtype(y.dtype, np.integer):
        raise ValueError("Labels must be a one-dimensional integer array")
    if not np.isfinite(p).all() or (p < 0).any() or (p > 1).any():
        raise ValueError("Probabilities must be finite and within [0, 1]")
    if not np.allclose(p.sum(1), 1, atol=1e-5, rtol=0):
        raise ValueError("Every probability row must sum to one")
    if (y < 0).any() or (y >= p.shape[1]).any():
        raise ValueError("Label index is outside the class range")
    return p, y


def reliability_bins(probs, labels, n_bins=15):
    p, y = validate(probs, labels)
    if not isinstance(n_bins, int) or n_bins < 1:
        raise ValueError("n_bins must be a positive integer")
    confidence = p.max(1)
    correct = (p.argmax(1) == y).astype(float)
    # [lo, hi), except the last bin includes confidence = 1 exactly.
    index = np.minimum((confidence * n_bins).astype(int), n_bins - 1)
    rows = []
    for i in range(n_bins):
        mask = index == i
        n = int(mask.sum())
        rows.append({"lower": i / n_bins, "upper": (i + 1) / n_bins,
                     "count": n,
                     "confidence": float(confidence[mask].mean()) if n else None,
                     "accuracy": float(correct[mask].mean()) if n else None})
    return rows


def classification_metrics(probs, labels, n_bins=15):
    p, y = validate(probs, labels)
    bins = reliability_bins(p, y, n_bins)
    ece = sum(b["count"] / len(y) * abs(b["accuracy"] - b["confidence"])
              for b in bins if b["count"])
    # Natural log, clamp only to make exact zero probabilities finite.
    nll = -np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)).mean()
    one_hot = np.eye(p.shape[1])[y]
    return {"accuracy": float((p.argmax(1) == y).mean()), "ece": float(ece),
            "nll": float(nll), "brier": float(np.square(p - one_hot).sum(1).mean()),
            "n": len(y), "ece_bins": n_bins}


def average_probabilities(sample_probs):
    p = np.asarray(sample_probs, dtype=np.float64)
    if p.ndim != 3 or len(p) == 0:
        raise ValueError("Expected samples by examples by classes")
    for sample in p:
        validate(sample, np.zeros(len(sample), dtype=int))
    return p.mean(axis=0)
''',
'src/ivon_lora/model.py': r'''"""Qwen option-token classification with LoRA on q_proj and v_proj."""
import torch
import torch.nn.functional as F
from peft import LoraConfig, get_peft_model


def option_token_ids(tokenizer):
    ids = [tokenizer.encode(" " + letter, add_special_tokens=False) for letter in "ABCD"]
    if any(len(x) != 1 for x in ids) or len({x[0] for x in ids}) != 4:
        raise ValueError("The tokenizer must encode ' A', ' B', ' C', ' D' as four single tokens")
    return [x[0] for x in ids]


def attach_lora(base, config):
    model = get_peft_model(base, LoraConfig(
        r=config["lora_rank"], lora_alpha=config["lora_alpha"],
        lora_dropout=config["lora_dropout"], bias="none", task_type="CAUSAL_LM",
        target_modules=config["target_modules"]))
    for name, p in model.named_parameters():
        if p.requires_grad:
            if "lora_" not in name:
                raise RuntimeError(f"Unexpected trainable base parameter: {name}")
            p.data = p.data.float()
    model.config.use_cache = False
    return model


def option_logits(model, batch, token_ids):
    base = model.get_base_model()
    # Qwen2Model contains the injected LoRA layers. Project only the final hidden
    # state onto four frozen vocabulary rows instead of allocating B*L*152k logits.
    # This is algebraically identical to selecting these logits from the full LM.
    if base.config.model_type != "qwen2":
        raise ValueError("This memory-saving path is validated for Qwen2 only")
    out = base.model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                     use_cache=False, return_dict=True)
    last = batch["attention_mask"].sum(1) - 1
    hidden = out.last_hidden_state[torch.arange(len(last), device=last.device), last]
    return F.linear(hidden, base.lm_head.weight[token_ids]).float()
''',
'src/ivon_lora/report.py': r'''"""Recompute every metric from saved predictions and create tables and plots."""
import argparse
import csv
import json
from pathlib import Path

import numpy as np

from .metrics import classification_metrics, reliability_bins

METHODS = ["adamw", "ivon_mean", "ivon_mc"]
METRICS = ["accuracy", "ece", "nll", "brier"]
LABELS = {"adamw": "AdamW-LoRA", "ivon_mean": "IVON @ mean", "ivon_mc": "IVON, 10 samples"}


def mean_std(values):
    values = np.asarray(values, dtype=float)
    return {"mean": float(values.mean()),
            "std": float(values.std(ddof=1)) if len(values) > 1 else None,
            "n_seeds": len(values)}


def paired_deltas(records, metric, method):
    base = {r["seed"]: r[metric] for r in records if r["method"] == "adamw"}
    other = {r["seed"]: r[metric] for r in records if r["method"] == method}
    if not base or base.keys() != other.keys():
        raise ValueError("Paired comparisons require identical seed sets")
    return mean_std([other[s] - base[s] for s in sorted(base)])


def build_report(root):
    root = Path(root)
    config = json.loads((root / "config.json").read_text())
    expected = set(config["seeds"])
    records, predictions, completed = [], {}, {}
    for seed in sorted(expected):
        completed[seed] = {}
        for opt in ["adamw", "ivon"]:
            path = root / f"seed_{seed}" / opt
            if not (path / "complete.json").exists():
                raise ValueError(f"Missing completed run: {path}. Partial runs are never labeled complete.")
            info = json.loads((path / "complete.json").read_text())
            completed[seed][opt] = info
            for method in (["adamw"] if opt == "adamw" else ["ivon_mean", "ivon_mc"]):
                pred = json.loads((path / f"test_{method}_predictions.json").read_text())
                values = classification_metrics(pred["probabilities"], np.array(pred["labels"], dtype=int), config["ece_bins"])
                saved = json.loads((path / f"test_{method}.json").read_text())
                for metric in METRICS:
                    if not np.isclose(values[metric], saved[metric], atol=1e-10, rtol=0):
                        raise ValueError(f"Saved metric does not match predictions: {path}, {metric}")
                records.append({"seed": seed, "method": method, **values,
                    "train_seconds": info["train_seconds"], "prediction_seconds": saved["prediction_seconds"]})
                predictions[(seed, method)] = pred
        if completed[seed]["adamw"]["initial_adapter_sha256"] != completed[seed]["ivon"]["initial_adapter_sha256"]:
            raise ValueError(f"Initial adapter mismatch for paired seed {seed}")
    reference = next(iter(predictions.values()))
    for pred in predictions.values():
        if pred["ids"] != reference["ids"] or pred["labels"] != reference["labels"]:
            raise ValueError("Test examples or labels differ across methods/seeds")
    dest = root / "summary"
    dest.mkdir(exist_ok=True)
    def save_csv(name, rows):
        with (dest / name).open("w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
    save_csv("per_seed.csv", records)
    summary = []
    for method in METHODS:
        row = {"method": method, "n_seeds": len(expected)}
        for metric in METRICS + ["train_seconds", "prediction_seconds"]:
            stats = mean_std([r[metric] for r in records if r["method"] == method])
            row[metric + "_mean"] = stats["mean"]
            row[metric + "_std"] = stats["std"]
        summary.append(row)
    save_csv("comparison.csv", summary)
    deltas = [{"method": method, "metric": metric, **paired_deltas(records, metric, method)}
              for method in METHODS[1:] for metric in METRICS]
    save_csv("paired_deltas.csv", deltas)
    payload = {"status": "measured", "model": config["model_id"],
        "dataset": config["dataset_config"], "config": config,
        "n_test": len(reference["labels"]), "summary": summary, "paired_deltas": deltas,
        "note": "Mean and sample SD across seeds. Delta = IVON minus AdamW. No significance claim from three seeds."}
    (dest / "summary.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
    make_plots(dest, records, predictions, config)
    # Readable terminal output, also captured in the Colab notebook.
    print("\nMEASURED TEST RESULTS (mean +/- sample SD across seeds)")
    for row in summary:
        def val(m, scale=1):
            sd = row[m + "_std"]
            return f"{scale * row[m + '_mean']:.3f} +/- {scale * sd:.3f}" if sd is not None else f"{scale * row[m + '_mean']:.3f} (one seed)"
        print(f"{row['method']:12} accuracy %: {val('accuracy',100)}   ECE pp: {val('ece',100)}   NLL: {val('nll')}")
    return payload


def make_plots(dest, records, predictions, config):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    colors = ["#43566e", "#108078", "#b15a35"]
    plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.7), constrained_layout=True)
    for ax, metric, title, factor in zip(axes, METRICS[:3], ["Accuracy (%)", "ECE (percentage points)", "NLL (natural log)"], [100,100,1]):
        for i, method in enumerate(METHODS):
            values = np.array([r[metric] * factor for r in records if r["method"] == method])
            ax.errorbar(i, values.mean(), yerr=values.std(ddof=1) if len(values) > 1 else None,
                        color=colors[i], fmt="o", capsize=5, markersize=8)
            ax.scatter(i + np.linspace(-.07,.07,len(values)), values, color=colors[i], s=18, alpha=.6)
        ax.set_xticks(range(3), ["AdamW", "IVON mean", "IVON MC"])
        ax.set_ylabel(title)
        ax.set_xlabel("Prediction method")
        ax.set_title(title + (" | higher is better" if metric == "accuracy" else " | lower is better"), fontsize=10)
    fig.suptitle(f"{config['model_id']} on {config['dataset_config']} test subset\nMeasured seed points, mean and sample SD", fontsize=12)
    fig.savefig(dest / "comparison.png", dpi=180)
    plt.close(fig)
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.9), constrained_layout=True)
    # Reliability diagrams show a single explicitly labeled seed. Pooling the
    # predictions across seeds would evaluate a different predictor/estimand.
    seed = config["seeds"][0]
    for ax, method, color in zip(axes, METHODS, colors):
        pred = predictions[(seed, method)]
        bins = reliability_bins(pred["probabilities"], np.array(pred["labels"],dtype=int), config["ece_bins"])
        populated = [b for b in bins if b["count"]]
        ax.plot([0,1], [0,1], "--", color="gray", label="Perfect calibration")
        ax.scatter([b["confidence"] for b in populated], [b["accuracy"] for b in populated],
                   s=[15 + b["count"] * 2 for b in populated], color=color, label="Observed bins")
        ax.set(xlim=(0,1), ylim=(0,1), xlabel="Mean confidence (probability)",
               ylabel="Empirical accuracy (fraction)", title=method)
        ax.legend(fontsize=7)
    fig.suptitle(f"Reliability on the held-out test subset, seed {seed}\n{config['ece_bins']} equal-width bins. Marker area increases with bin count.", fontsize=12)
    fig.savefig(dest / "reliability.png", dpi=180)
    plt.close(fig)
    # ECE bin sensitivity is descriptive, never a search for the best-looking ECE.
    sensitivity = []
    for (seed, method), pred in predictions.items():
        for n_bins in [5,10,15,20]:
            m = classification_metrics(pred["probabilities"], np.array(pred["labels"],dtype=int), n_bins)
            sensitivity.append({"seed":seed,"method":method,"bins":n_bins,"ece":m["ece"]})
    with (dest / "ece_sensitivity.csv").open("w",newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(sensitivity[0]))
        writer.writeheader()
        writer.writerows(sensitivity)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("run_dir")
    build_report(parser.parse_args().run_dir)
''',
'src/ivon_lora/train.py': r'''"""Paired-seed experiments, atomic result files, and complete provenance."""
import argparse
from contextlib import nullcontext
import gc
import hashlib
import importlib.metadata
import json
import math
import os
from pathlib import Path
import random
import subprocess
import time

import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import HfApi
from ivon import IVON
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer

from .data import Collator, prepare_splits
from .metrics import classification_metrics
from .model import attach_lora, option_logits, option_token_ids


def load_config(path):
    path = Path(path)
    config = json.loads(path.read_text())
    if "extends" in config:
        parent = load_config(path.parent / config.pop("extends"))
        parent.update(config)
        config = parent
    for key in ["steps", "batch_size", "accumulation_steps", "posterior_samples", "ece_bins"]:
        if not isinstance(config[key], int) or config[key] <= 0:
            raise ValueError(f"{key} must be a positive integer")
    if not config["seeds"] or len(set(config["seeds"])) != len(config["seeds"]):
        raise ValueError("Provide distinct seeds")
    if config["dtype"] not in ["float32", "bfloat16"]:
        raise ValueError("Use float32 on T4, or bfloat16 on a supported GPU")
    return config


def write_json(path, content):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(content, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    # Reproducibility is expected on the same environment. Cross-device bitwise
    # equivalence is not claimed. No stochastic dropout is used in the defaults.


def to_device(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def make_optimizer(model, method, config):
    params = [p for p in model.parameters() if p.requires_grad]
    if method == "adamw":
        return torch.optim.AdamW(params, lr=config["adamw_lr"], weight_decay=config["weight_decay"])
    if method != "ivon":
        raise ValueError(method)
    return IVON(params, lr=config["ivon_lr"], ess=config["ess"],
                hess_init=config["hess_init"], weight_decay=config["weight_decay"],
                clip_radius=config["clip_radius"], beta2=config["beta2"], rescale_lr=False)


def train_step(model, optimizer, microbatches, token_ids, method):
    optimizer.zero_grad(set_to_none=True)
    n = sum(len(b["labels"]) for b in microbatches)
    loss_value = 0.0
    # One posterior draw remains active across all accumulation microbatches.
    # IVON collects the final gradient on context exit. Never clip or unscale
    # gradients after that point: IVON has already copied them into its state.
    with optimizer.sampled_params(train=True) if method == "ivon" else nullcontext():
        for batch in microbatches:
            loss = F.cross_entropy(option_logits(model, batch, token_ids), batch["labels"], reduction="sum") / n
            if not torch.isfinite(loss):
                raise FloatingPointError("Nonfinite loss. The run has been stopped.")
            loss.backward()
            loss_value += loss.detach().item()
        if any(p.grad is None or not torch.isfinite(p.grad).all()
               for p in model.parameters() if p.requires_grad):
            raise FloatingPointError("Missing or nonfinite adapter gradients")
    optimizer.step()
    return loss_value


@torch.inference_mode()
def predict(model, loader, token_ids, device):
    model.eval()
    parts = []
    for batch in loader:
        logits = option_logits(model, to_device(batch, device), token_ids)
        parts.append(logits.softmax(-1).double().cpu().numpy())
    return np.concatenate(parts)


def evaluate(model, optimizer, method, loader, token_ids, device, samples, seed):
    if method != "ivon_mc":
        return predict(model, loader, token_ids, device)
    # A fixed draw spans the entire evaluation set. Average probabilities,
    # not logits or weights. fork_rng prevents evaluation changing train RNG.
    devices = [torch.cuda.current_device()] if device == "cuda" else []
    with torch.random.fork_rng(devices=devices):
        torch.manual_seed(seed)
        total = None
        for sample in range(samples):
            with optimizer.sampled_params(train=False):
                p = predict(model, loader, token_ids, device)
            total = p if total is None else total + p
            print(f"  posterior draw {sample + 1}/{samples}", flush=True)
    return total / samples


def source_hash():
    digest = hashlib.sha256()
    for path in sorted(Path(__file__).parent.glob("*.py")):
        digest.update(path.name.encode())
        digest.update(path.read_bytes())
    return digest.hexdigest()


def sync(device):
    if device == "cuda":
        torch.cuda.synchronize()


def run(config, out_dir, device):
    root = Path(out_dir)
    root.mkdir(parents=True, exist_ok=True)
    if device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("No CUDA GPU. Select Runtime > Change runtime type > T4 GPU in Colab.")
    if config["dtype"] == "bfloat16" and (device != "cuda" or not torch.cuda.is_bf16_supported()):
        raise RuntimeError("bfloat16 requires a supported GPU. T4 must use float32.")
    # Resolve mutable hub refs once, then pin downloads to immutable commits.
    api = HfApi()
    config = dict(config)
    config["model_revision"] = api.model_info(config["model_id"], revision=config.get("model_revision")).sha
    config["dataset_revision"] = api.dataset_info(config["dataset_id"], revision=config.get("dataset_revision")).sha
    config["source_sha256"] = source_hash()
    config_path = root / "config.json"
    if config_path.exists() and json.loads(config_path.read_text()) != config:
        raise ValueError("Output already belongs to a different config/source revision. Use a new --output directory.")
    write_json(config_path, config)
    tokenizer = AutoTokenizer.from_pretrained(config["model_id"], revision=config["model_revision"])
    tokenizer.pad_token = tokenizer.eos_token
    token_ids = option_token_ids(tokenizer)
    raw = load_dataset(config["dataset_id"], config["dataset_config"], revision=config["dataset_revision"])
    splits, manifest = prepare_splits(raw, tokenizer, config)
    manifest["dataset_id"] = config["dataset_id"]
    manifest["dataset_revision"] = config["dataset_revision"]
    write_json(root / "split_manifest.json", manifest)
    environment = {"python": os.sys.version, "torch": torch.__version__,
        "device": torch.cuda.get_device_name(0) if device == "cuda" else "CPU",
        "cuda": torch.version.cuda, "option_token_ids": token_ids,
        "packages": {p: importlib.metadata.version(p) for p in
            ["transformers", "peft", "ivon-opt", "datasets", "numpy"]}}
    write_json(root / "environment.json", environment)
    collator = Collator(tokenizer.pad_token_id)
    loaders = {split: DataLoader(rows, batch_size=config["eval_batch_size"],
        collate_fn=collator, shuffle=False) for split, rows in splits.items() if split != "train"}
    for seed in config["seeds"]:
        for method in ["adamw", "ivon"]:
            target = root / f"seed_{seed}" / method
            if (target / "complete.json").exists():
                print(f"Already complete: {target}", flush=True)
                continue
            target.mkdir(parents=True, exist_ok=True)
            seed_all(seed)
            dtype = getattr(torch, config["dtype"])
            base = AutoModelForCausalLM.from_pretrained(config["model_id"],
                revision=config["model_revision"], torch_dtype=dtype, attn_implementation="sdpa")
            model = attach_lora(base, config).to(device)
            del base
            trainable = {name: p for name, p in model.named_parameters() if p.requires_grad}
            initial_hash = hashlib.sha256(b"".join(p.detach().cpu().numpy().tobytes()
                                                    for p in trainable.values())).hexdigest()
            optimizer = make_optimizer(model, method, config)
            generator = torch.Generator().manual_seed(seed)
            loader = DataLoader(splits["train"], batch_size=config["batch_size"],
                shuffle=True, collate_fn=collator, generator=generator)
            iterator = iter(loader)
            if device == "cuda":
                torch.cuda.reset_peak_memory_stats()
            initial_lr = config[f"{method}_lr"]
            warmup = max(1, math.ceil(config["steps"] * config["warmup_fraction"]))
            history = []
            print(f"START {method} seed={seed}, {config['steps']} steps, {sum(p.numel() for p in trainable.values()):,} trainable parameters", flush=True)
            sync(device)
            started = time.perf_counter()
            for step in range(config["steps"]):
                scale = (step + 1) / warmup if step < warmup else (config["steps"] - step) / max(1, config["steps"] - warmup)
                for group in optimizer.param_groups:
                    group["lr"] = initial_lr * scale
                microbatches = []
                for _ in range(config["accumulation_steps"]):
                    try:
                        batch = next(iterator)
                    except StopIteration:
                        iterator = iter(loader)
                        batch = next(iterator)
                    microbatches.append(to_device(batch, device))
                model.train()
                loss = train_step(model, optimizer, microbatches, token_ids, method)
                history.append({"step": step + 1, "loss": loss, "lr": initial_lr * scale})
                if step == 0 or (step + 1) % 16 == 0:
                    print(f"  step {step + 1}/{config['steps']} loss={loss:.4f}", flush=True)
            sync(device)
            train_seconds = time.perf_counter() - started
            peak = torch.cuda.max_memory_allocated() / 1024**3 if device == "cuda" else None
            write_json(target / "training_history.json", history)
            run_info = {"seed": seed, "optimizer": method, "initial_adapter_sha256": initial_hash,
                "trainable_parameters": sum(p.numel() for p in trainable.values()),
                "total_parameters": sum(p.numel() for p in model.parameters()),
                "train_seconds": train_seconds, "peak_train_memory_gib": peak,
                "steps": config["steps"], "status": "measured", "results": []}
            if config["save_checkpoints"]:
                model.save_pretrained(target / "adapter")
                tokenizer.save_pretrained(target / "adapter")
                # IVON state stores the posterior Hessian; mean adapters alone
                # cannot reproduce posterior-sampling predictions.
                torch.save({"optimizer": optimizer.state_dict(),
                            "ivon_current_step": getattr(optimizer, "current_step", None),
                            "trainable_parameter_names": list(trainable)}, target / "optimizer.pt")
            for split in ["validation", "test"]:
                labels = np.array([r["label"] for r in splits[split]], dtype=int)
                names = ["adamw"] if method == "adamw" else ["ivon_mean", "ivon_mc"]
                for prediction_method in names:
                    sync(device)
                    eval_start = time.perf_counter()
                    probs = evaluate(model, optimizer, prediction_method, loaders[split], token_ids,
                        device, config["posterior_samples"], seed + 10000 + (split == "test"))
                    sync(device)
                    metrics = classification_metrics(probs, labels, config["ece_bins"])
                    metrics.update({"method": prediction_method, "split": split, "seed": seed,
                        "prediction_seconds": time.perf_counter() - eval_start,
                        "posterior_samples": config["posterior_samples"] if prediction_method == "ivon_mc" else 0})
                    write_json(target / f"{split}_{prediction_method}.json", metrics)
                    # Dataset text is not redistributed. IDs map to the cited ARC dataset.
                    prediction_file = target / f"{split}_{prediction_method}_predictions.json"
                    write_json(prediction_file, {"ids": [r["id"] for r in splits[split]],
                                                "labels": labels.tolist(), "probabilities": probs.tolist()})
                    run_info["results"].append(metrics)
                    print(json.dumps(metrics), flush=True)
            write_json(target / "complete.json", run_info)
            del model, optimizer, trainable, microbatches
            gc.collect()
            if device == "cuda":
                torch.cuda.empty_cache()
    print("ALL RUNS COMPLETE. Generating summary.", flush=True)
    from .report import build_report
    build_report(root)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", default="configs/quick.json")
    parser.add_argument("--output", default="runs/quick")
    parser.add_argument("--device", choices=["cuda", "cpu"], default="cuda")
    args = parser.parse_args()
    run(load_config(args.config), args.output, args.device)


if __name__ == "__main__":
    main()
''',
'tests/test_metrics.py': r'''import math
import numpy as np
import pytest
from ivon_lora.metrics import classification_metrics, reliability_bins, average_probabilities
from ivon_lora.report import paired_deltas


def test_known_calibration_example():
    # Both confidences are .8, exactly one is correct: ECE = .3.
    result = classification_metrics([[.8,.2],[.2,.8]], np.array([0,0]))
    assert result["accuracy"] == .5
    assert result["ece"] == pytest.approx(.3)
    assert result["nll"] == pytest.approx(-.5*(math.log(.8)+math.log(.2)))
    assert result["brier"] == pytest.approx(.68)


def test_confidence_one_is_included():
    p = np.eye(3)
    result = classification_metrics(p, np.array([0,1,2]))
    assert result["accuracy"] == 1
    assert result["ece"] == 0
    assert result["nll"] == 0
    assert sum(b["count"] for b in reliability_bins(p,np.array([0,1,2]))) == 3


@pytest.mark.parametrize("p,y", [([[.8,.8]],[0]), ([[float('nan'),0]],[0]), ([[.2,.8]],[2]), ([],[])])
def test_invalid_probabilities_rejected(p,y):
    with pytest.raises(ValueError):
        classification_metrics(p,np.array(y,dtype=int))


def test_probability_ensemble_and_paired_seeds():
    np.testing.assert_allclose(average_probabilities([[[.9,.1]],[[.3,.7]]]), [[.6,.4]])
    records = [{"seed":21,"method":"adamw","ece":.3}, {"seed":42,"method":"adamw","ece":.4},
               {"seed":21,"method":"ivon_mc","ece":.2},{"seed":42,"method":"ivon_mc","ece":.2}]
    assert paired_deltas(records,"ece","ivon_mc")["mean"] == pytest.approx(-.15)
    with pytest.raises(ValueError):
        paired_deltas(records[:-1],"ece","ivon_mc")
''',
'tests/test_training.py': r'''"""Offline checks on a randomly initialized tiny Qwen. These are not research results."""
import copy
import numpy as np
import pytest
import torch
from transformers import Qwen2Config, Qwen2ForCausalLM
from ivon_lora.data import Collator, format_arc
from ivon_lora.model import attach_lora, option_logits
from ivon_lora.train import make_optimizer, train_step, load_config


@pytest.fixture
def setup():
    torch.set_num_threads(2)
    torch.manual_seed(21)
    conf = load_config("configs/quick.json")
    base = Qwen2ForCausalLM(Qwen2Config(vocab_size=64, hidden_size=32,
        intermediate_size=64, num_hidden_layers=2, num_attention_heads=4,
        num_key_value_heads=2, max_position_embeddings=64, attention_dropout=0.0))
    model = attach_lora(base,conf)
    batch = Collator(0)([{"input_ids":[2,3,4],"label":0},{"input_ids":[2,5],"label":2}])
    return model,batch,conf


def test_option_logits_match_full_model_with_padding(setup):
    model,batch,_ = setup
    model.eval()
    ids = [6,7,8,9]
    direct = option_logits(model,batch,ids)
    full = model(input_ids=batch["input_ids"],attention_mask=batch["attention_mask"],use_cache=False).logits
    selected = full[torch.arange(2),batch["attention_mask"].sum(1)-1][:,ids]
    torch.testing.assert_close(direct,selected,atol=1e-6,rtol=1e-5)


@pytest.mark.parametrize("method",["adamw","ivon"])
def test_only_adapters_change_and_posterior_restores(setup,method):
    model,batch,conf = setup
    frozen = {n:p.detach().clone() for n,p in model.named_parameters() if not p.requires_grad}
    before = [p.detach().clone() for p in model.parameters() if p.requires_grad]
    opt = make_optimizer(model,method,conf)
    loss = train_step(model,opt,[batch], [6,7,8,9],method)
    assert np.isfinite(loss)
    after = [p.detach().clone() for p in model.parameters() if p.requires_grad]
    assert any(not torch.equal(a,b) for a,b in zip(before,after))
    for n,p in model.named_parameters():
        if n in frozen:
            torch.testing.assert_close(frozen[n],p,rtol=0,atol=0)
    if method == "ivon":
        with opt.sampled_params(train=False):
            assert any(not torch.equal(a,p) for a,p in zip(after,[p for p in model.parameters() if p.requires_grad]))
        for a,p in zip(after,[p for p in model.parameters() if p.requires_grad]):
            torch.testing.assert_close(a,p,rtol=0,atol=0)
        restored = make_optimizer(model,method,conf)
        restored.load_state_dict(copy.deepcopy(opt.state_dict()))
        assert torch.all(restored.param_groups[0]["hess"] > 0)


def test_accumulated_gradient_matches_full_batch(setup):
    model,batch,conf = setup
    model2 = copy.deepcopy(model)
    opt1,opt2 = make_optimizer(model,"ivon",conf),make_optimizer(model2,"ivon",conf)
    torch.manual_seed(99)
    train_step(model,opt1,[batch],[6,7,8,9],"ivon")
    torch.manual_seed(99)
    train_step(model2,opt2,[{k:v[:1] for k,v in batch.items()},{k:v[1:] for k,v in batch.items()}],[6,7,8,9],"ivon")
    for a,b in zip(model.parameters(),model2.parameters()):
        torch.testing.assert_close(a,b,atol=1e-6,rtol=1e-5)


def test_numeric_answer_keys_and_no_label_leak():
    row = {"id":"x","question":"Which option?","choices":{"label":["1","2","3","4"],"text":["one","two","three","four"]},"answerKey":"3"}
    item = format_arc(row)
    assert item["label"] == 2
    assert item["prompt"].endswith("Answer:")
    row["answerKey"] = "1"
    assert format_arc(row)["prompt"] == item["prompt"]
''',
}
for relative, content in FILES.items():
    target = ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding='utf-8')
os.chdir(ROOT)
print('Project source ready:', len(FILES), 'files')


## 1. Environment and offline correctness checks
PyTorch comes from Colab. The code uses FP32 on T4 to keep IVON gradient and posterior updates numerically straightforward. The final-token projection avoids allocating full-vocabulary logits.

In [ ]:
# Stream subprocess output into Colab and retain a text log for the presentation.
import time
Path('execution_logs').mkdir(exist_ok=True)
def run_logged(label, command):
    print('Starting:', label, flush=True)
    started = time.perf_counter()
    with open('execution_logs/' + label + '.log', 'a') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
    print(label, 'elapsed seconds:', round(time.perf_counter()-started, 1), flush=True)
    if code:
        raise RuntimeError(label + ' failed; read the error above or its execution log.')
run_logged('gpu_check', [sys.executable, '-u', '-c',
    "import torch; print('PyTorch:',torch.__version__); assert torch.cuda.is_available(), 'Select T4 GPU, not TPU'; print('GPU:',torch.cuda.get_device_name(0))"])
# Install in a subprocess so already-imported notebook packages do not interfere.
deps = [line for line in Path('requirements.txt').read_text().splitlines()
        if line and not line.startswith(('#', 'torch'))]
run_logged('install', [sys.executable, '-m', 'pip', 'install', '-q', *deps])
run_logged('project_install', [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
run_logged('correctness_tests', [sys.executable, '-u', '-m', 'pytest', '-q'])


## 2. Actual-model smoke run
This downloads the public model and ARC data, runs two optimizer steps per method, then checks evaluation and reporting. Its metrics are debugging evidence only.

In [ ]:
run_logged('gpu_smoke', [sys.executable, '-u', '-m', 'ivon_lora.train', '--config', 'configs/smoke.json', '--output', 'runs/smoke'])


## 3. Main experiment
The quick profile trains both optimizers for 128 steps on the same 512 training questions, for seeds 21, 42, and 87. Validation uses 128 questions and the held-out test uses 256. IVON predictions include the posterior mean and a 10-sample probability average.

Do not change the configuration after inspecting test metrics. Use `configs/standard.json` and a new output directory for the larger extension. A completed run resumes by skipping finished optimizer/seed pairs. An interrupted pair restarts from initialization.

Runtime depends on Colab availability and GPU speed. Keep this tab open. Results in `/content` are temporary until you download the archive below.


The default `registered_t4` profile preserves the exact model and dataset revisions and hyperparameters of the completed T4 experiment. Runtime package versions are recorded with every run; cross-environment bitwise identity is not assumed.

In [ ]:
PROFILE = 'registered_t4'
run_logged('three_seed_training', [sys.executable, '-u', '-m', 'ivon_lora.train', '--config', f'configs/{PROFILE}.json', '--output', f'runs/{PROFILE}'])


## 4. Measured results and calibration
The table gives mean and **sample standard deviation**, not standard error. ECE in the CSV is a fraction; multiply by 100 to obtain percentage points. Deltas are IVON minus AdamW. Three seeds offer limited evidence about statistical significance.

In [ ]:
import pandas as pd
from IPython.display import display, Image
summary = Path(f'runs/{PROFILE}/summary')
run_logged('training_figures', [sys.executable, 'scripts/training_figures.py', f'runs/{PROFILE}'])
display(pd.read_csv(summary / 'comparison.csv'))
display(pd.read_csv(summary / 'paired_deltas.csv'))
display(pd.read_csv(summary / 'training_details.csv'))
display(Image(filename=str(summary / 'training_curves.png')))
display(Image(filename=str(summary / 'comparison.png')))
display(Image(filename=str(summary / 'reliability.png')))


## 5. Download the evidence
This archive contains predictions, seed metrics, data IDs, model and dataset revision hashes, training logs and small adapter checkpoints. The base model is not included. Save the executed notebook with **File > Download > .ipynb** as well.

In [ ]:
import shutil
from google.colab import files
shutil.copytree('execution_logs', ROOT / f'runs/{PROFILE}/execution_logs', dirs_exist_ok=True)
archive = shutil.make_archive('/content/ivon_lora_measured_results', 'zip', ROOT / f'runs/{PROFILE}')
files.download(archive)
print('Completed experiment archive:', archive)
print('Also save this executed notebook: File > Download > .ipynb')


## Interpretation for the presentation
- If ECE and NLL decrease across paired seeds, describe evidence of better calibration **in this experiment**.
- If only accuracy improves, do not claim improved calibration.
- If ECE decreases but NLL worsens, discuss the disagreement and show the bin sensitivity file.
- If the gap is small relative to seed SD, call the result inconclusive.
- Qwen differs from Llama in model family and pretraining as well as size. This study cannot isolate a pure model-size effect.
- These are probabilities normalized over four answer tokens, not unrestricted generation confidence.
- The original paper's 2.8-point accuracy and 4.6-point ECE improvements describe **IVON @ mean**. They are not this project's results.
